# Combined refactoring and checkpoint-evaluation results

This notebook combines the 26-August refactoring result files with the 27-August DOMINANT checkpoint evaluation. It adds model and source-file provenance, validates the row keys, and writes one analysis-ready CSV.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_ROOT = ROOT / 'artifacts' / 'results'
LOW_DATA_RESULTS_ROOT = RESULTS_ROOT / 'low_data_exps'
EXPERIMENT_DATE = '2026-08-26'
OUTPUT_CSV = LOW_DATA_RESULTS_ROOT / f'refactoring_{EXPERIMENT_DATE}_joint.csv'
DOMINANT_RESULT_FILE = LOW_DATA_RESULTS_ROOT / 'dominant_checkpoint_0_1_percent.csv'
EXPECTED_MODELS = {'anemone', 'cola', 'dominant', 'gadnr', 'gradate'}

In [2]:
refactoring_files = sorted(LOW_DATA_RESULTS_ROOT.glob(f'*_refactoring_{EXPERIMENT_DATE}/results.csv'))
if not refactoring_files:
    raise FileNotFoundError(f'No refactoring result files found for {EXPERIMENT_DATE}')
if not DOMINANT_RESULT_FILE.is_file():
    raise FileNotFoundError(f'Missing DOMINANT checkpoint-evaluation results: {DOMINANT_RESULT_FILE}')

result_sources = [
    (
        result_file.parent.name.removesuffix(f'_refactoring_{EXPERIMENT_DATE}'),
        EXPERIMENT_DATE,
        result_file,
    )
    for result_file in refactoring_files
] + [('dominant', '2026-08-27', DOMINANT_RESULT_FILE)]

frames = []
for model, experiment_date, result_file in result_sources:
    frame = pd.read_csv(result_file)
    frame.insert(0, 'model', model)
    frame.insert(1, 'experiment_date', experiment_date)
    frame['source_file'] = str(result_file.relative_to(ROOT))
    frames.append(frame)

joint_results = pd.concat(frames, ignore_index=True)
models_found = set(joint_results['model'])
missing_models = EXPECTED_MODELS - models_found
if missing_models:
    raise ValueError(f'Missing expected models: {sorted(missing_models)}')

key_columns = ['model', 'dataset', 'seed', 'split']
duplicate_rows = joint_results.duplicated(key_columns, keep=False)
if duplicate_rows.any():
    raise ValueError('Duplicate experiment rows found:\n' + joint_results.loc[duplicate_rows, key_columns].to_string(index=False))
if not joint_results['status'].eq('ok').all():
    raise ValueError('At least one experiment row does not have status=ok')

joint_results = joint_results.sort_values(key_columns).reset_index(drop=True)
joint_results.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(joint_results):,} rows from {len(result_sources)} files to {OUTPUT_CSV.relative_to(ROOT)}')
joint_results.head()

Wrote 150 rows from 5 files to artifacts/results/low_data_exps/refactoring_2026-08-26_joint.csv


,model,experiment_date,dataset,seed,split,threshold,train_macro_f1,validation_macro_f1,validation_auc,test_macro_f1,test_auc,training_seconds,status,source_file
0,anemone,2026-08-26,UAE,1,0,0.735720,0.494949,0.451561,0.426128,0.464772,0.432805,77.865812,ok,artifacts/results/low_data_exps/anemone_refact...
1,anemone,2026-08-26,UAE,2,0,0.713837,0.600000,0.361545,0.427956,0.346627,0.411938,330.392701,ok,artifacts/results/low_data_exps/anemone_refact...
2,anemone,2026-08-26,UAE,3,0,0.772843,0.523810,0.440201,0.410379,0.428250,0.402413,320.550529,ok,artifacts/results/low_data_exps/anemone_refact...
3,anemone,2026-08-26,UAE,4,0,0.757271,0.583333,0.454124,0.430869,0.453453,0.427718,335.111333,ok,artifacts/results/low_data_exps/anemone_refact...
4,anemone,2026-08-26,UAE,5,0,0.776142,0.494949,0.448213,0.415941,0.456729,0.429921,314.057066,ok,artifacts/results/low_data_exps/anemone_refact...


In [3]:
summary = (
    joint_results.groupby(['model', 'dataset'], as_index=False)
    .agg(
        seeds=('seed', 'nunique'),
        test_macro_f1_mean=('test_macro_f1', 'mean'),
        test_macro_f1_std=('test_macro_f1', 'std'),
        test_auc_mean=('test_auc', 'mean'),
        test_auc_std=('test_auc', 'std'),
        training_seconds_mean=('training_seconds', 'mean'),
    )
    .sort_values(['dataset', 'model'])
)
summary

,model,dataset,seeds,test_macro_f1_mean,test_macro_f1_std,test_auc_mean,test_auc_std,training_seconds_mean
0,anemone,UAE,5,0.429966,0.048548,0.420959,0.013150,275.595488
6,cola,UAE,5,0.662768,0.036788,0.774373,0.025485,12.691768
12,dominant,UAE,5,0.779976,0.077807,0.927512,0.003198,0.000000
18,gadnr,UAE,5,0.755149,0.037392,0.836561,0.005333,104.691440
24,gradate,UAE,5,0.475114,0.028612,0.424383,0.022217,650.678368
1,anemone,china,5,0.514260,0.042574,0.710437,0.008417,258.929194
7,cola,china,5,0.483697,0.032804,0.622827,0.035809,14.615762
13,dominant,china,5,0.532041,0.078473,0.698459,0.019212,0.000000
19,gadnr,china,5,0.489934,0.012103,0.522029,0.051857,42.809707
25,gradate,china,5,0.517144,0.028529,0.660246,0.017105,1450.235009
